# OptiMystic JupyterLab Integration Test Notebook

This notebook is for testing and debugging, not for Voila web deployment.

It lets you run and debug each layer independently in JupyterLab:
- Python-only (`cp`)
- Julia-only (`mip`, called through the Python CLI)
- R bridge connectivity (`rpy2` + `r_solvers`)
- Full pipeline (Python/Julia -> R post-processing)

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "examples").exists() and (PROJECT_ROOT.parent / "examples").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "examples"))
from jupyter_debug_tools import (
    run_python_only_cp_scheduling,
    run_julia_only_mip_packing,
    run_full_pipeline,
    ensure_r_bridge,
    run_r_postprocess,
    explain_debug_pipeline,
    run_section,
)

print("Project root:", PROJECT_ROOT)

## What Each Section Tests

- Python-only: validates only Python CP logic
- Julia-only: validates only Julia MIP execution path
- R bridge: validates only `rpy2` and `r_solvers` connectivity
- Full pipeline: validates end-to-end handoff from Python/Julia outputs to R post-processing

This structure helps isolate failures quickly by layer.

In [ ]:
print(json.dumps(explain_debug_pipeline(), ensure_ascii=False, indent=2))

# Quick targeted run: "python" | "julia" | "full"
SECTION = "python"
section_result = run_section(SECTION)
print(f"Section={SECTION}")
print(json.dumps({k: v.get('status') for k, v in section_result.items()}, indent=2))

In [ ]:
# 1) Python-only
python_cp_result = run_python_only_cp_scheduling()
print(json.dumps({
    "status": python_cp_result.get("status"),
    "objective": python_cp_result.get("objective"),
    "variables_count": len(python_cp_result.get("variables", [])),
}, indent=2))

In [ ]:
# 2) Julia-only (via python cli -> julia)
julia_mip_result = run_julia_only_mip_packing()
print(json.dumps({
    "status": julia_mip_result.get("status"),
    "objective": julia_mip_result.get("objective"),
    "variables_count": len(julia_mip_result.get("variables", [])),
}, indent=2))

In [ ]:
# 3) R bridge check
r_info = ensure_r_bridge()
print(json.dumps(r_info, indent=2))

In [ ]:
# 4) Full pipeline -> R post-processing
pipeline = run_full_pipeline()

packing_store = {
    "parameters": {
        "Items": ["A", "B", "C"],
        "Weights": [2, 3, 4],
        "Values": [10, 12, 14],
        "Capacity": 7,
    }
}
packing_processed = run_r_postprocess("packing", pipeline["julia_mip"], packing_store)

print(json.dumps({
    "python_cp_status": pipeline["python_cp"].get("status"),
    "julia_mip_status": pipeline["julia_mip"].get("status"),
    "r_processed_mode": packing_processed.get("mode"),
    "r_processed_status": packing_processed.get("status"),
}, indent=2))

In [ ]:
# 5) R sensitivity test (new)
sensitivity_input = {
    "lp_sensitivity": True,
    "constraints": [
        {"Constraint": "C_0", "Shadow Price": 3.5, "Slack": 0.0},
        {"Constraint": "C_1", "Shadow Price": -1.2, "Slack": 1.0},
        {"Constraint": "capacity", "Shadow Price": 0.8, "Slack": 2.0},
    ],
}
sensitivity_store = {
    "parameters": {
        "Items": ["ItemA", "ItemB"]
    }
}

import importlib
ro = importlib.import_module("rpy2.robjects")
res_json = json.dumps(sensitivity_input).replace("'", "\\'")
store_json = json.dumps(sensitivity_store).replace("'", "\\'")
ro.r(f"sens_res <- jsonlite::fromJSON('{res_json}', simplifyVector = FALSE)")
ro.r(f"sens_store <- jsonlite::fromJSON('{store_json}', simplifyVector = FALSE)")
ro.r("sens_out <- process_sensitivity(sens_res, sens_store, 'cutting')")
sens_json = str(ro.r("jsonlite::toJSON(sens_out, auto_unbox = TRUE, null = 'null')")[0])
print(json.dumps(json.loads(sens_json), indent=2))

In [ ]:
# 6) Domain-specific sensitivity checks (scheduling + resourcing)
import importlib
ro = importlib.import_module("rpy2.robjects")

sched_sens = {
    "lp_sensitivity": True,
    "constraints": [
        {"Constraint": "coverage_Morning", "Shadow Price": 2.1, "Slack": 0.0},
        {"Constraint": "maxShift_E1", "Shadow Price": 0.7, "Slack": 0.0},
        {"Constraint": "maxShift_E2", "Shadow Price": 0.1, "Slack": 1.0},
    ],
}
res_sens = {
    "lp_sensitivity": True,
    "constraints": [
        {"Constraint": "cpu_capacity", "Shadow Price": 4.0, "Slack": 0.0},
        {"Constraint": "ram_capacity", "Shadow Price": 3.2, "Slack": 0.0},
        {"Constraint": "demand_bound", "Shadow Price": 0.3, "Slack": 2.0},
    ],
}

ro.r(f"sched_sens <- jsonlite::fromJSON('{json.dumps(sched_sens).replace("'", "\\'")}', simplifyVector = FALSE)")
ro.r("sched_out <- process_sensitivity(sched_sens, list(parameters=list()), 'scheduling')")
ro.r(f"res_sens <- jsonlite::fromJSON('{json.dumps(res_sens).replace("'", "\\'")}', simplifyVector = FALSE)")
ro.r("res_out <- process_sensitivity(res_sens, list(parameters=list()), 'resourcing')")

sched_out_json = str(ro.r("jsonlite::toJSON(sched_out, auto_unbox = TRUE, null = 'null')")[0])
res_out_json = str(ro.r("jsonlite::toJSON(res_out, auto_unbox = TRUE, null = 'null')")[0])

print("Scheduling sensitivity insight:")
print(json.loads(sched_out_json)["insight"])
print("\nResourcing sensitivity insight:")
print(json.loads(res_out_json)["insight"])

In [ ]:
assert pipeline["python_cp"].get("status") not in ("Error", "error")
assert pipeline["julia_mip"].get("status") not in ("Error", "error")
assert packing_processed.get("status") == "ok"
print("All targeted tests passed (Python/Julia/R).")